# Mid-Atlantic Gas Merchant: storage-year LP

One storage year (April 2025 – March 2026) for a gas merchant buying at Eastern Gas South and serving NY, NJ and PA.
This notebook loads the processed data, validates it, solves the LP in `model.py`, checks the solution, and reports the plan and its shadow prices.

Run top to bottom (*Runtime → Run all* in Colab). Tables only; figures come in Milestone 5.

In [1]:
# Setup. In Colab: clone the repo and install gurobipy only (numpy, pandas and
# matplotlib are preinstalled; reinstalling pinned versions can break the live
# runtime). Locally: install requirements.txt yourself and run from the repo root.
import os, sys, subprocess

REPO_URL = "https://github.com/robjiakai/MidAtlanticGasMerchant.git"
REPO_DIR = "MidAtlanticGasMerchant"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gurobipy==13.0.3"], check=True)

if not os.path.isfile("model.py"):
    raise RuntimeError("Run this notebook from the repository root (the folder containing model.py).")

for path in (os.path.abspath("."), os.path.abspath("src")):
    if path not in sys.path:
        sys.path.insert(0, path)

import gurobipy as gp
import pandas as pd
pd.set_option("display.float_format", "{:,.2f}".format)
print("Working directory:", os.getcwd())
print("Gurobi", ".".join(map(str, gp.gurobi.version())))

Working directory: /Users/robli/Documents/Columbia/2026Fall/Optimization/MidAtlanticGasMerchant
Gurobi 13.0.3


## 1. Load and validate inputs

`load_data()` enforces the CSV schema and calls `validate_inputs()`; it raises if anything is wrong.
Accessible pipeline capacity (`kbar`) is not in the data files: it comes from the `capacity_limits` argument, whose default is the Milestone 3 baseline (assumption A9). Fuel retention is 0 on every route (A11).

In [2]:
import warnings
from load_data import load_data
from validate import validate_solution
from model import solve

CAPACITY_LIMITS = {1: 19178, 2: 19178, 3: 21308, 4: 25570}   # Dth/day, baseline (A9)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Pipeline upload has no access shares")
    data = load_data(capacity_limits=CAPACITY_LIMITS)

T, J, L, F = data["T"], data["J"], data["L"], data["F"]
MONTHS = ["Apr-25", "May-25", "Jun-25", "Jul-25", "Aug-25", "Sep-25",
          "Oct-25", "Nov-25", "Dec-25", "Jan-26", "Feb-26", "Mar-26"]
ROUTE_NAME = {1: "Empire → NY", 2: "NFG → NY", 3: "TGP → NJ", 4: "TETCO → PA"}
print(f"Loaded {len(T)} months, {len(J)} markets, {len(L)} routes, {len(F)} storage fields.")

Loaded 12 months, 3 markets, 4 routes, 2 storage fields.


### Input summary

In [3]:
routes_in = pd.DataFrame({
    "route": [ROUTE_NAME[l] for l in L],
    "usage c ($/Dth)": [data["c"][l] for l in L],
    "reservation R ($/Dth/d/mo)": [data["R"][l] for l in L],
    "accessible kbar (Dth/d)": [data["kbar"][l] for l in L],
    "fuel phi": [data["phi"][l] for l in L],
}, index=pd.Index(L, name="l"))

storage_in = pd.DataFrame({
    "capacity Ibar (Dth)": data["Ibar"], "opening I0 (Dth)": data["I0"],
    "max inj (Dth/d)": data["r_in"], "max wd (Dth/d)": data["r_out"],
    "inj fee ($/Dth)": data["c_in"], "wd fee ($/Dth)": data["c_out"],
}).rename_axis("f")

monthly_in = pd.DataFrame({
    "days": [data["days"][t] for t in T],
    "hub P": [data["P"][t] for t in T],
    **{f"citygate {j}": [data["pi"][j, t] for t in T] for j in J},
    **{f"demand {j}": [data["D"][j, t] for t in T] for j in J},
}, index=pd.Index(MONTHS, name="month"))

display(routes_in, storage_in, monthly_in)

,route,usage c ($/Dth),reservation R ($/Dth/d/mo),accessible kbar (Dth/d),fuel phi
l,,,,,
1,Empire → NY,0.00,7.00,"19,178.00",0.00
2,NFG → NY,0.01,6.13,"19,178.00",0.00
3,TGP → NJ,0.05,4.67,"21,308.00",0.00
4,TETCO → PA,0.12,15.63,"25,570.00",0.00


,capacity Ibar (Dth),opening I0 (Dth),max inj (Dth/d),max wd (Dth/d),inj fee ($/Dth),wd fee ($/Dth)
f,,,,,,
H,"2,210,500.00","657,453.00","12,280.00","50,054.00",0.04,0.04
M,"1,298,303.00","436,473.00","12,983.00","25,966.00",0.04,0.04


,days,hub P,citygate NY,citygate NJ,citygate PA,demand NY,demand NJ,demand PA
month,,,,,,,,
Apr-25,30,3.16,4.88,5.66,4.66,"935,268.00","645,626.00","654,062.00"
May-25,31,2.34,5.63,5.00,5.37,"744,006.00","326,279.00","648,819.00"
Jun-25,30,2.09,6.92,4.36,5.72,"831,634.00","423,533.00","682,009.00"
Jul-25,31,2.03,7.32,4.56,5.04,"1,002,410.00","520,244.00","755,462.00"
Aug-25,31,2.29,7.24,4.68,6.05,"799,823.00","418,575.00","685,874.00"
Sep-25,30,1.78,6.83,4.86,7.05,"697,189.00","361,460.00","617,296.00"
Oct-25,31,1.62,4.70,5.05,6.15,"839,410.00","462,109.00","619,773.00"
Nov-25,30,2.63,4.53,5.40,5.81,"1,256,698.00","688,585.00","759,718.00"
Dec-25,31,3.79,4.74,5.49,4.93,"1,829,858.00","1,019,073.00","991,786.00"


## 2. Solve

Set `VERBOSE = True` to print the Gurobi log.

In [4]:
VERBOSE = False
sol = solve(data, verbose=VERBOSE)
m = sol.get("model")
print("Status:", sol["status"])
if sol["status"] != "OPTIMAL":
    print("IIS:", sol.get("iis"))
    raise RuntimeError("Model did not solve to optimality.")
print(f"Size: {m.NumVars} variables, {m.NumConstrs} constraints")
print(f"Total annual cost: ${sol['objective']:,.0f}")

costs = pd.Series(sol["costs"], name="$")
costs = pd.concat([costs, pd.Series({"total": costs.sum()}, name="$")])
pd.DataFrame({"$": costs, "share": costs / sol["objective"]}).style.format({"$": "${:,.0f}", "share": "{:.1%}"})

Set parameter Username
Set parameter LicenseID to value 2869026
Academic license - for non-commercial use only - expires 2027-09-28
Status: OPTIMAL
Size: 172 variables, 198 constraints
Total annual cost: $103,971,660


,$,share
purchase,"$66,019,178",63.5%
pipeline_usage,"$1,307,463",1.3%
storage_fees,"$327,047",0.3%
spot,"$27,305,185",26.3%
reservation,"$9,012,787",8.7%
total,"$103,971,660",100.0%


## 3. Check the solution

`validate_solution()` recomputes every constraint residual and cost term from the solution values. An empty list means the solution is feasible and the reported costs add up.

In [5]:
problems = validate_solution(sol, data)
assert not problems, "Solution failed validation (see printed problems)"
print("Solution passes all feasibility and cost checks.")

Solution passes all feasibility and cost checks.


## 4. Monthly plan

**Supply and storage.** Hub purchases, shipments by route (measured at the hub), and storage injections, withdrawals and end-of-month inventory.

In [6]:
plan = pd.DataFrame({
    "hub P": [data["P"][t] for t in T],
    "hub buy x": [sol["x"][t] for t in T],
    **{f"ship {ROUTE_NAME[l]}": [sol["s"][l, t] for t in T] for l in L},
    **{f"inj {f}": [sol["qin"][f, t] for t in T] for f in F},
    **{f"wd {f}": [sol["qout"][f, t] for t in T] for f in F},
    **{f"inv {f}": [sol["I"][f, t] for t in T] for f in F},
}, index=pd.Index(MONTHS, name="month"))
plan.style.format("{:,.0f}").format({"hub P": "{:.2f}"})

,hub P,hub buy x,ship Empire → NY,ship NFG → NY,ship TGP → NJ,ship TETCO → PA,inj H,inj M,wd H,wd M,inv H,inv M
month,,,,,,,,,,,,
Apr-25,3.16,1134644.000000,498795.000000,0.000000,639240.000000,654062.000000,0.000000,0.000000,657453.000000,436473.000000,0.000000,0.000000
May-25,2.34,2050764.000000,594518.000000,149488.000000,326279.000000,648819.000000,331660.000000,0.000000,0.000000,0.000000,331660.000000,0.000000
Jun-25,2.09,2624622.000000,575340.000000,575340.000000,423533.000000,682009.000000,368400.000000,319046.000000,0.000000,0.000000,700060.000000,319046.000000
Jul-25,2.03,2845422.000000,594518.000000,594518.000000,520244.000000,755462.000000,380680.000000,186626.000000,0.000000,0.000000,1080740.000000,505672.000000
Aug-25,2.29,2338467.000000,594518.000000,258820.000000,418575.000000,685874.000000,380680.000000,53515.000000,0.000000,0.000000,1461420.000000,559187.000000
Sep-25,1.78,2433835.000000,575340.000000,511339.000000,361460.000000,617296.000000,368400.000000,389490.000000,0.000000,0.000000,1829820.000000,948677.000000
Oct-25,1.62,2651598.000000,594518.000000,594518.000000,462109.000000,619773.000000,380680.000000,349626.000000,0.000000,0.000000,2210500.000000,1298303.000000
Nov-25,2.63,2549638.000000,575340.000000,575340.000000,639240.000000,759718.000000,0.000000,0.000000,0.000000,0.000000,2210500.000000,1298303.000000
Dec-25,3.79,2642254.000000,594518.000000,594518.000000,660548.000000,792670.000000,0.000000,0.000000,0.000000,0.000000,2210500.000000,1298303.000000


**How each market's demand is met.** Pipeline deliveries (net of fuel), net withdrawals from market-area storage, and citygate purchases. They sum to demand in every month (constraint DM).

In [7]:
rows = []
for j in J:
    for t, month in zip(T, MONTHS):
        pipe = sum((1 - data["phi"][l]) * sol["s"][l, t] for l in data["L_j"][j])
        stor = sum(sol["qout"][f, t] - sol["qin"][f, t] for f in data["F_j"][j])
        rows.append({"market": j, "month": month, "pipeline": pipe, "storage (net)": stor,
                     "citygate": sol["u"][j, t], "demand": data["D"][j, t],
                     "citygate price": data["pi"][j, t]})
supply = pd.DataFrame(rows).set_index(["market", "month"])
supply["citygate share"] = supply["citygate"] / supply["demand"]

annual = supply.groupby(level="market", sort=False)[["pipeline", "storage (net)", "citygate", "demand"]].sum()
annual["citygate share"] = annual["citygate"] / annual["demand"]
display(annual.style.format({c: "{:,.0f}" for c in annual.columns[:4]} | {"citygate share": "{:.1%}"}))
supply.round(2)

,pipeline,storage (net),citygate,demand,citygate share
market,,,,,
NY,"12,234,246",0,"1,765,255","13,999,501",12.6%
NJ,"6,368,948",0,"1,408,553","7,777,501",18.1%
PA,"7,801,023",0,"1,531,976","9,332,999",16.4%


pipeline  storage (net)   citygate       demand  \
market month                                                        
NY     Apr-25   498,795.00     436,473.00       0.00   935,268.00   
       May-25   744,006.00           0.00       0.00   744,006.00   
       Jun-25 1,150,680.00    -319,046.00       0.00   831,634.00   
       Jul-25 1,189,036.00    -186,626.00       0.00 1,002,410.00   
       Aug-25   853,338.00     -53,515.00       0.00   799,823.00   
       Sep-25 1,086,679.00    -389,490.00       0.00   697,189.00   
       Oct-25 1,189,036.00    -349,626.00       0.00   839,410.00   
       Nov-25 1,150,680.00           0.00 106,018.00 1,256,698.00   
       Dec-25 1,189,036.00           0.00 640,822.00 1,829,858.00   
       Jan-26 1,189,036.00     804,946.00     888.00 1,994,870.00   
       Feb-26   804,888.00     459,357.00 423,362.00 1,687,607.00   
       Mar-26 1,189,036.00    -402,473.00 594,165.00 1,380,728.00   
NJ     Apr-25   639,240.00           0.00   6,386.00   645,626.00   
       May-25   326,279.00           0.00       0.00   326,279.00   
       Jun-25   423,533.00           0.00       0.00   423,533.00   
       Jul-25   520,244.00           0.00       0.00   520,244.00   
       Aug-25   418,575.00           0.00       0.00   418,575.00   
       Sep-25   361,460.00           0.00       0.00   361,460.00   
       Oct-25   462,109.00           0.00       0.00   462,109.00   
       Nov-25   639,240.00           0.00  49,345.00   688,585.00   
       Dec-25   660,548.00           0.00 358,525.00 1,019,073.00   
       Jan-26   660,548.00           0.00 430,047.00 1,090,595.00   
       Feb-26   596,624.00           0.00 506,674.00 1,103,298.00   
       Mar-26   660,548.00           0.00  57,576.00   718,124.00   
PA     Apr-25   654,062.00           0.00       0.00   654,062.00   
       May-25   648,819.00           0.00       0.00   648,819.00   
       Jun-25   682,009.00           0.00       0.00   682,009.00   
       Jul-25   755,462.00           0.00       0.00   755,462.00   
       Aug-25   685,874.00           0.00       0.00   685,874.00   
       Sep-25   617,296.00           0.00       0.00   617,296.00   
       Oct-25   619,773.00           0.00       0.00   619,773.00   
       Nov-25   759,718.00           0.00       0.00   759,718.00   
       Dec-25   792,670.00           0.00 199,116.00   991,786.00   
       Jan-26   792,670.00           0.00 346,464.00 1,139,134.00   
       Feb-26         0.00           0.00 968,503.00   968,503.00   
       Mar-26   792,670.00           0.00  17,893.00   810,563.00   

               citygate price  citygate share  
market month                                   
NY     Apr-25            4.88            0.00  
       May-25            5.63            0.00  
       Jun-25            6.92            0.00  
       Jul-25            7.32            0.00  
       Aug-25            7.24            0.00  
       Sep-25            6.83            0.00  
       Oct-25            4.70            0.00  
       Nov-25            4.53            0.08  
       Dec-25            4.74            0.35  
       Jan-26           10.03            0.00  
       Feb-26            6.09            0.25  
       Mar-26            4.97            0.43  
NJ     Apr-25            5.66            0.01  
       May-25            5.00            0.00  
       Jun-25            4.36            0.00  
       Jul-25            4.56            0.00  
       Aug-25            4.68            0.00  
       Sep-25            4.86            0.00  
       Oct-25            5.05            0.00  
       Nov-25            5.40            0.07  
       Dec-25            5.49            0.35  
       Jan-26            6.04            0.39  
       Feb-26            7.13            0.46  
       Mar-26            6.43            0.08  
PA     Apr-25            4.66            0.00  
       May-25            5.37            0.00  
       Jun-25            5.72            0.00  
   

## 5. Capacity and shadow prices

Duals as returned by `model.py`, read as the change in total annual cost from relaxing one constraint by one unit:

- **KC** (accessible capacity, reported as a positive saving): annual value of one more Dth/day of accessible capacity on route ℓ, **net** of its reservation charge, which is already in the objective. The gross value of the capacity is therefore KC + 12R. Contracting more is worthwhile whenever KC > 0.
- **SL_cap** (storage space, reported as a positive saving): value of one more Dth of working-gas capacity in field f at the end of month t.
- **DM** (market balance): marginal cost of serving one more Dth in market j, month t.

In [8]:
capacity = pd.DataFrame({
    "route": [ROUTE_NAME[l] for l in L],
    "contracted k (Dth/d)": [sol["k"][l] for l in L],
    "accessible kbar (Dth/d)": [data["kbar"][l] for l in L],
    "reservation 12R ($/yr)": [12 * data["R"][l] for l in L],
    "KC dual, net ($/yr)": [sol["duals"]["KC"][l] for l in L],
}, index=pd.Index(L, name="l"))
capacity["gross value KC + 12R ($/yr)"] = capacity["KC dual, net ($/yr)"] + capacity["reservation 12R ($/yr)"]
capacity["avg load factor"] = [
    sum(sol["s"][l, t] for t in T) / (sum(data["days"][t] for t in T) * sol["k"][l]) if sol["k"][l] > 0 else float("nan")
    for l in L]
display(capacity.style.format({"avg load factor": "{:.0%}"}, precision=2, thousands=","))

storage_dual = pd.DataFrame({f: [sol["duals"]["SL_cap"][f, t] for t in T] for f in F},
                            index=pd.Index(MONTHS, name="month"))
delivered = pd.DataFrame({j: [sol["duals"]["DM"][j, t] for t in T] for j in J},
                         index=pd.Index(MONTHS, name="month"))
storage_dual = storage_dual.where(storage_dual.abs() > 1e-9, 0.0)   # clear -0.00
display(storage_dual.add_prefix("SL_cap ").round(4), delivered.add_prefix("DM ").round(2))

,route,contracted k (Dth/d),accessible kbar (Dth/d),reservation 12R ($/yr),"KC dual, net ($/yr)",gross value KC + 12R ($/yr),avg load factor
l,,,,,,,
1,Empire → NY,"19,178.00","19,178.00",84.04,304.75,388.79,99%
2,NFG → NY,"19,178.00","19,178.00",73.62,313.16,386.78,76%
3,TGP → NJ,"21,308.00","21,308.00",56.00,361.87,417.88,82%
4,TETCO → PA,"25,570.00","25,570.00",187.56,11.09,198.65,84%


,SL_cap H,SL_cap M
month,,
Apr-25,0.00,0.00
May-25,0.00,0.00
Jun-25,0.00,0.00
Jul-25,0.00,0.00
Aug-25,0.00,0.00
Sep-25,0.00,0.00
Oct-25,0.29,2.16
Nov-25,1.08,0.21
Dec-25,0.27,1.35


,DM NY,DM NJ,DM PA
month,,,
Apr-25,3.16,5.66,3.28
May-25,2.35,2.39,2.46
Jun-25,2.30,2.14,2.21
Jul-25,2.30,2.08,2.15
Aug-25,2.30,2.34,2.41
Sep-25,1.79,1.83,1.90
Oct-25,2.30,1.67,1.74
Nov-25,4.53,5.40,2.75
Dec-25,4.74,5.49,4.93


## 6. Binding constraints

Inequality constraints (PC, KC, SL_in, SL_out, SL_cap) with zero slack and a nonzero dual; equality constraints always hold with equality, so they are left out. Duals here are Gurobi's raw values: for a ≤ constraint in a minimization they are ≤ 0, and their magnitude is the saving from one more unit. These are the bottlenecks that drive the plan, and the first places to look in sensitivity analysis.

In [9]:
binding = [
    {"constraint": c.ConstrName, "family": c.ConstrName.split("[")[0], "Gurobi dual": c.Pi}
    for c in m.getConstrs()
    if c.Sense != "=" and abs(c.Slack) <= 1e-6 and abs(c.Pi) > 1e-9   # equalities always bind
]
binding = pd.DataFrame(binding)
display(binding.groupby("family").size().rename("binding count").to_frame())
binding.set_index("constraint")

,binding count
family,
KC,4
PC,27
SL_cap,6
SL_in,8
SL_out,2


,family,Gurobi dual
constraint,,
"PC[1,2]",PC,-0.01
"PC[1,3]",PC,-0.21
"PC[1,4]",PC,-0.27
"PC[1,5]",PC,-0.01
"PC[1,6]",PC,-0.01
"PC[1,7]",PC,-0.68
"PC[1,8]",PC,-1.90
"PC[1,9]",PC,-0.95
"PC[1,10]",PC,-5.97


## 7. Save outputs

In [ ]:
os.makedirs("outputs", exist_ok=True)
plan.to_csv("outputs/monthly_plan.csv")
supply.to_csv("outputs/market_supply.csv")
capacity.to_csv("outputs/capacity_duals.csv")
delivered.to_csv("outputs/marginal_cost_by_market.csv")
costs.to_csv("outputs/cost_breakdown.csv", header=True)
print("Wrote monthly_plan, market_supply, capacity_duals, marginal_cost_by_market, cost_breakdown to outputs/")